[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/signal38/signal38.github.io/blob/main/notebooks/00_acled_labels.ipynb)

# Notebook 00 — ACLED Ground Truth Labels

Fetches North Korean conflict events from the ACLED API and derives escalation labels grounded in human-coded severity data. Where ACLED coverage exists, these labels replace Claude-distilled labels. Where ACLED has no events for a given week, the Claude label is retained as fallback.

**Prerequisites:**
- An ACLED account with API access ([register here](https://acleddata.com/register/))
- Two Colab secrets (key icon in the left sidebar):
  - `ACLED_EMAIL` — your ACLED account email
  - `ACLED_PASSWORD` — your ACLED API key (from your [account page](https://acleddata.com/acleddatanew/wp-content/uploads/dlm_uploads/2023/06/ACLED-Access-Guide-2023.pdf))
  - `GITHUB_TOKEN` — PAT with write access to the signal38 org

**Outputs (published to `main`):**
- `data/acled/nk_events.jsonl` — raw ACLED events for North Korea
- `data/labeled/all_labeled.jsonl` — merged labels with `label_source` field

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
print(f'Repo ready at {REPO}')

In [ ]:
# Install requests if not already present (it usually is on Colab)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'requests'], check=True)
print('Dependencies ready.')

In [ ]:
# ── Credentials from Colab secrets ────────────────────────────────────────
# Add these via the key icon (🔑) in the Colab left sidebar before running.
# Never paste credentials directly into a cell.

from google.colab import userdata

ACLED_EMAIL    = userdata.get('ACLED_EMAIL')
ACLED_PASSWORD = userdata.get('ACLED_PASSWORD')

if not ACLED_EMAIL or not ACLED_PASSWORD:
    raise RuntimeError(
        'Missing ACLED credentials. Add ACLED_EMAIL and ACLED_PASSWORD '
        'via the key icon in the Colab left sidebar.'
    )

print(f'Credentials loaded for: {ACLED_EMAIL}')


In [ ]:
import requests
import json
import time
from datetime import datetime, timedelta

ACLED_LOGIN_URL = 'https://acleddata.com/user/login?_format=json'
ACLED_API       = 'https://acleddata.com/api/acled/read'

FIELDS = 'event_id_cnty|event_date|event_type|sub_event_type|actor1|actor2|location|fatalities|notes|source'


def make_acled_session(email: str, password: str) -> requests.Session:
    """Log in and return an authenticated session."""
    session = requests.Session()
    resp = session.post(ACLED_LOGIN_URL, json={'name': email, 'pass': password}, timeout=30)
    resp.raise_for_status()
    print('Logged in to ACLED.')
    return session


def fetch_acled_nk(session: requests.Session, page_size: int = 500) -> list[dict]:
    """Fetch all ACLED events for North Korea, paginated."""
    all_events = []
    page = 1

    while True:
        params = {
            'country': 'North Korea',
            'fields':  FIELDS,
            'limit':   page_size,
            'page':    page,
        }
        resp = session.get(ACLED_API, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        events = data.get('data', [])
        if not events:
            break

        all_events.extend(events)
        print(f'  Page {page}: {len(events)} events (total: {len(all_events)})')

        if len(events) < page_size:
            break

        page += 1
        time.sleep(0.5)

    return all_events


session  = make_acled_session(ACLED_EMAIL, ACLED_PASSWORD)
nk_events = fetch_acled_nk(session)
print(f'
Total NK events fetched: {len(nk_events)}')


In [ ]:
# ── Inspect event type distribution ───────────────────────────────────────
from collections import Counter

type_counts = Counter(e['event_type'] for e in nk_events)
subtype_counts = Counter(e['sub_event_type'] for e in nk_events)

print('Event types:')
for t, n in type_counts.most_common():
    print(f'  {t}: {n}')

print('\nSub-event types (top 10):')
for t, n in subtype_counts.most_common(10):
    print(f'  {t}: {n}')

# Date range
dates = sorted(e['event_date'] for e in nk_events)
print(f'\nDate range: {dates[0]} → {dates[-1]}')

# Total fatalities
total_fatalities = sum(int(e.get('fatalities', 0) or 0) for e in nk_events)
print(f'Total fatalities recorded: {total_fatalities}')

In [ ]:
# ── Save raw ACLED events ──────────────────────────────────────────────────
acled_out = PATHS['data_root'] / 'acled'
acled_out.mkdir(parents=True, exist_ok=True)
acled_path = acled_out / 'nk_events.jsonl'

with open(acled_path, 'w') as f:
    for e in nk_events:
        f.write(json.dumps(e) + '\n')

print(f'Saved {len(nk_events)} events → {acled_path}')

In [ ]:
# ── Derive escalation level from ACLED event mix ───────────────────────────
#
# Mapping rationale:
#   Nuclear/missile tests are existential signals regardless of fatalities.
#   Battles and remote violence escalate based on fatality count.
#   Strategic developments (exercises, statements) map to 2-3.
#   Riots/protests within NK are internal and map to 1-2.
#
# When a week has multiple events, we take the maximum level across all events
# (escalation level = worst event that week).

NUCLEAR_KEYWORDS = [
    'nuclear', 'missile', 'icbm', 'ballistic', 'rocket', 'warhead',
    'hydrogen', 'bomb test', 'underground test', 'launch',
]


def event_to_escalation(event: dict) -> int:
    """Map a single ACLED event to an escalation level (1-5)."""
    etype    = (event.get('event_type')     or '').lower()
    subtype  = (event.get('sub_event_type') or '').lower()
    notes    = (event.get('notes')          or '').lower()
    fatals   = int(event.get('fatalities', 0) or 0)

    # Nuclear / missile tests — always critical regardless of fatalities
    if any(kw in notes or kw in subtype for kw in NUCLEAR_KEYWORDS):
        return 5

    if etype == 'battles':
        return 5 if fatals > 0 else 4

    if etype == 'explosions/remote violence':
        return 5 if fatals > 0 else 4

    if etype == 'violence against civilians':
        return 4 if fatals > 0 else 3

    if etype == 'strategic developments':
        # Missile/weapons-adjacent strategic events
        if any(kw in notes for kw in ['military', 'exercise', 'drill', 'mobiliz']):
            return 3
        return 2

    if etype in ('riots', 'protests'):
        return 1

    return 2  # conservative default


def week_start(date_str: str) -> str:
    """Return ISO week start (Sunday) for a given ACLED date string (YYYY-MM-DD)."""
    d = datetime.strptime(date_str, '%Y-%m-%d')
    # GDELT uses week_start as the Monday; align to same convention
    monday = d - timedelta(days=d.weekday())
    return monday.strftime('%Y-%m-%d')


# Group events by week, take max escalation per week
from collections import defaultdict

acled_by_week: dict[str, list[dict]] = defaultdict(list)
for e in nk_events:
    w = week_start(e['event_date'])
    acled_by_week[w].append(e)

acled_escalation: dict[str, dict] = {}
for w, events in acled_by_week.items():
    levels = [event_to_escalation(e) for e in events]
    acled_escalation[w] = {
        'escalation_level': max(levels),
        'event_count':      len(events),
        'event_types':      list({e['event_type'] for e in events}),
        'total_fatalities': sum(int(e.get('fatalities', 0) or 0) for e in events),
    }

print(f'ACLED weeks with coverage: {len(acled_escalation)}')
level_dist = Counter(v['escalation_level'] for v in acled_escalation.values())
print('Escalation level distribution:')
for lvl in sorted(level_dist):
    print(f'  Level {lvl}: {level_dist[lvl]} weeks')

In [ ]:
# ── Merge with Claude labels ───────────────────────────────────────────────
# Priority: ACLED (human-coded) > Claude (distilled)
# All entries get a label_source field for traceability.

labeled_path = PATHS['labeled']
claude_labels = []
with open(labeled_path) as f:
    for line in f:
        if line.strip():
            claude_labels.append(json.loads(line))

merged = []
acled_count  = 0
claude_count = 0

for entry in claude_labels:
    week = entry['week_start']
    if week in acled_escalation:
        # Override escalation level with ACLED-derived value
        acled_data = acled_escalation[week]
        entry = dict(entry)  # don't mutate original
        entry['assessment'] = dict(entry.get('assessment', {}))
        entry['assessment']['escalation_level'] = acled_data['escalation_level']
        entry['assessment']['escalation_rationale'] = (
            f"ACLED-derived: {acled_data['event_count']} event(s) coded "
            f"({', '.join(acled_data['event_types'])}), "
            f"{acled_data['total_fatalities']} fatalities recorded."
        )
        entry['label_source']     = 'acled'
        entry['acled_event_count']= acled_data['event_count']
        acled_count += 1
    else:
        entry = dict(entry)
        entry['label_source'] = 'claude'
        claude_count += 1

    merged.append(entry)

print(f'Total labeled weeks:  {len(merged)}')
print(f'  ACLED-grounded:     {acled_count} ({100*acled_count/len(merged):.1f}%)')
print(f'  Claude fallback:    {claude_count} ({100*claude_count/len(merged):.1f}%)')

In [ ]:
# ── Agreement analysis — how well did Claude track ACLED? ─────────────────
# On ACLED-covered weeks, compare Claude's original escalation to ACLED's.
# This validates the distillation methodology and goes in the paper.

agreements = []
for entry in merged:
    if entry['label_source'] != 'acled':
        continue
    week = entry['week_start']
    acled_level  = acled_escalation[week]['escalation_level']
    # Find original Claude label from raw labeled file
    # (we need to reload before the merge overwrote it)
    # Workaround: re-read original file
    agreements.append((week, acled_level))

# Re-read originals for comparison
claude_by_week = {}
with open(labeled_path) as f:
    for line in f:
        if line.strip():
            e = json.loads(line)
            lvl = e.get('assessment', {}).get('escalation_level')
            if lvl is not None:
                claude_by_week[e['week_start']] = int(lvl)

diffs = []
for week, acled_lvl in agreements:
    claude_lvl = claude_by_week.get(week)
    if claude_lvl is not None:
        diffs.append(abs(acled_lvl - claude_lvl))

if diffs:
    import numpy as np
    within_1 = sum(d <= 1 for d in diffs) / len(diffs)
    exact     = sum(d == 0 for d in diffs) / len(diffs)
    print(f'Claude vs ACLED agreement (on {len(diffs)} overlapping weeks):')
    print(f'  Exact match:       {exact:.1%}')
    print(f'  Within ±1 level:   {within_1:.1%}')
    print(f'  Mean |error|:      {np.mean(diffs):.2f}')
    print()
    print('  → Report these numbers in Section 4 (Evaluation Strategy).')
else:
    print('No overlapping weeks found — check date alignment between ACLED and clusters.')

In [ ]:
# ── Save merged labels ─────────────────────────────────────────────────────
with open(labeled_path, 'w') as f:
    for entry in merged:
        f.write(json.dumps(entry) + '\n')

print(f'Updated {labeled_path}')

# Also regenerate train/val/test splits to pick up the new labels
# (re-read training files and update escalation_level in assistant turns)
import re

label_map = {
    e['week_start']: e for e in merged
}

def update_split(split_name: str):
    path = PATHS['training_dir'] / f'{split_name}.jsonl'
    updated = []
    changed = 0
    with open(path) as f:
        for line in f:
            if not line.strip():
                continue
            example = json.loads(line)
            # Extract week from the user message content
            user_content = example['messages'][1]['content']
            week_match = re.search(r'Week: (\d{4}-\d{2}-\d{2})', user_content)
            if week_match:
                week = week_match.group(1)
                label_entry = label_map.get(week)
                if label_entry and label_entry['label_source'] == 'acled':
                    # Update escalation_level in the assistant message JSON
                    asst_content = example['messages'][2]['content']
                    try:
                        asst_json = json.loads(asst_content)
                        old_level = asst_json.get('escalation_level')
                        new_level = label_entry['assessment']['escalation_level']
                        if old_level != new_level:
                            asst_json['escalation_level'] = new_level
                            asst_json['escalation_rationale'] = label_entry['assessment']['escalation_rationale']
                            example = dict(example)
                            example['messages'] = list(example['messages'])
                            example['messages'][2] = dict(example['messages'][2])
                            example['messages'][2]['content'] = json.dumps(asst_json)
                            changed += 1
                    except (json.JSONDecodeError, TypeError):
                        pass
            updated.append(example)

    with open(path, 'w') as f:
        for ex in updated:
            f.write(json.dumps(ex) + '\n')

    print(f'  {split_name}.jsonl: {changed} examples updated with ACLED labels')

print('Updating training splits...')
update_split('train')
update_split('val')
update_split('test')

In [ ]:
# ── Publish to GitHub ──────────────────────────────────────────────────────
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=[
        'data/acled/nk_events.jsonl',
        'data/labeled/all_labeled.jsonl',
        'data/training/train.jsonl',
        'data/training/val.jsonl',
        'data/training/test.jsonl',
    ],
    message='Add ACLED ground truth labels for NK risk assessment',
    repo_dir=REPO,
)